## Dataset analysis: `student_sponsorships_list16.csv`

Purpose: sponsorship/scholarship disbursements (List16). Used to model sponsored students explicitly.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_sponsorships_list16.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df["DISBURSEMENT_DATE"] = pd.to_datetime(df["DISBURSEMENT_DATE"], errors="coerce")
df["AMOUNT_SPONSORED_UGX"] = pd.to_numeric(df["AMOUNT_SPONSORED_UGX"], errors="coerce")
df["STATUS"] = df["STATUS"].astype(str).str.upper().str.strip()

df.isna().mean().sort_values(ascending=False).head(30)

In [ ]:
df.duplicated(["SPONSOR_PAYMENT_ID"]).sum(), df["SPONSOR_PAYMENT_ID"].isna().sum()

In [ ]:
df["STATUS"].value_counts(dropna=False)

In [ ]:
df.groupby("STATUS").agg(
    events=("SPONSOR_PAYMENT_ID","count"),
    students=("REG_NO","nunique"),
    amount_sum=("AMOUNT_SPONSORED_UGX","sum"),
    amount_mean=("AMOUNT_SPONSORED_UGX","mean"),
).sort_values("events", ascending=False)

In [ ]:
df["SPONSOR_NAME"].value_counts().head(20)

## Advanced analytics

Focus: sponsorship intensity by semester and how it associates with CGPA (joined to transcript).

In [ ]:
import numpy as np
from analysis_utils import basic_profile, missingness_report, numeric_outlier_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

In [ ]:
numeric_outlier_report(df, cols=["AMOUNT_SPONSORED_UGX"]).head(20)

In [ ]:
trans = pd.read_csv(DATA_DIR / "student_transcript_list16.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

s = df.copy()
s["AMOUNT_SPONSORED_UGX"] = pd.to_numeric(s["AMOUNT_SPONSORED_UGX"], errors="coerce")
s["STATUS"] = s["STATUS"].astype(str).str.upper().str.strip()
s["_approved"] = s["STATUS"].isin(["APPROVED","PAID","SUCCESS"]).astype(int)

agg = s.groupby(["REG_NO","SEMESTER_INDEX"], as_index=False).agg(
    sponsor_event_count=("SPONSOR_PAYMENT_ID","count"),
    sponsor_amount_total=("AMOUNT_SPONSORED_UGX","sum"),
    sponsor_amount_approved=("AMOUNT_SPONSORED_UGX", lambda x: float(x[s.loc[x.index, "_approved"].astype(bool)].sum(skipna=True))),
    sponsor_distinct_sponsors=("SPONSOR_NAME","nunique"),
    sponsor_distinct_types=("SCHOLARSHIP_TYPE","nunique"),
)
agg["has_sponsorship"] = (agg["sponsor_event_count"]>0).astype(int)

joined = merge_to_transcript_for_cgpa(agg, trans, on=["REG_NO","SEMESTER_INDEX"], how="inner")
joined.groupby("has_sponsorship")["CGPA"].agg(["count","mean","median","std"])